# 06 — Customer Segmentation (k-means)

**Owner:** Person A (modelling track).

**Rubric line:** Depth via combining methods. This is the layer that
turns a black-box probability into actionable customer archetypes.


In [ ]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data, features, models, metrics, decision, viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 6.1 — Load split + the deployable feature set

In [ ]:
train_df, test_df = data.load_interim()
feature_cols = data.feature_columns(include_leaky=False)


## 6.2 — Build a clustering-ready feature matrix

Use the same preprocessor as the model so segments are defined on the
same feature space — but without the target.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

preproc = features.build_preprocessor(scale_numeric=True)
X_train_t = preproc.fit_transform(train_df[feature_cols])
X_test_t = preproc.transform(test_df[feature_cols])


## 6.3 — Choose k via elbow + silhouette

In [ ]:
inertias, silhouettes = [], []
for k in config.KMEANS_K_RANGE:
    km = KMeans(n_clusters=k, random_state=config.RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_train_t)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_train_t, labels, sample_size=5000, random_state=config.RANDOM_SEED))

diag = pd.DataFrame({'k': list(config.KMEANS_K_RANGE), 'inertia': inertias, 'silhouette': silhouettes})
diag


## 6.4 — Fit final k-means at the chosen k

Pick the k where the silhouette is high AND the elbow is visible.


In [ ]:
K = 5  # update after looking at the diagnostic above
kmeans = KMeans(n_clusters=K, random_state=config.RANDOM_SEED, n_init=10)
train_segments = kmeans.fit_predict(X_train_t)
test_segments = kmeans.predict(X_test_t)


## 6.5 — Profile each segment

For each segment: size, base subscription rate, average age, dominant job,
dominant contact channel. Build a one-line persona from this.


In [ ]:
train_with_seg = train_df.assign(segment=train_segments)
profile = (
    train_with_seg.groupby('segment')
    .agg(n=(config.TARGET_COL, 'size'),
         conv_rate=(config.TARGET_COL, 'mean'),
         avg_age=('age', 'mean'),
         dominant_job=('job', lambda s: s.mode().iloc[0]),
         dominant_contact=('contact', lambda s: s.mode().iloc[0]))
    .sort_values('conv_rate', ascending=False)
)
profile.to_csv(config.TABLES_DIR / 'segment_profiles.csv')
profile


## 6.6 — Score the model within each segment

Does the LightGBM model perform uniformly across segments, or are there
segments where it's notably weaker (high uncertainty → escalate to
human review)?


In [ ]:
import joblib
lgbm_artifact = joblib.load(config.MODELS_DIR / 'improved_lgbm.joblib')
lgbm_proba = lgbm_artifact['proba_test']
test_with_seg = test_df.assign(segment=test_segments, proba=lgbm_proba)

from sklearn.metrics import average_precision_score
perf_by_segment = (
    test_with_seg.groupby('segment')
    .apply(lambda g: pd.Series({
        'n': len(g),
        'base_rate': g[config.TARGET_COL].mean(),
        'pr_auc': average_precision_score(g[config.TARGET_COL], g['proba']) if g[config.TARGET_COL].nunique() > 1 else np.nan,
    }))
)
perf_by_segment.to_csv(config.TABLES_DIR / 'segment_model_perf.csv')
perf_by_segment


## 6.7 — Persona narratives (slide-ready)

One sentence per segment, e.g.:

- **Segment 0 — "Affluent retirees":** older, cellular contact, prior
  campaign success, base conv. rate 35% → high-touch retention focus.
- **Segment 1 — "Young blue-collar":** under 35, telephone contact,
  no prior contact, base 4% → deprioritise unless model gives a strong signal.
- ...
